In [2]:
import city2graph as c2g
import osmnx as ox
import duckdb
import geopandas as gpd
from shapely import wkb, Point
import matplotlib.pyplot as plt
ox.settings.overpass_url = 'https://overpass.kumi.systems/api'

In [3]:
from typing import NamedTuple


class BBox(NamedTuple):
    xmin: float
    ymin: float
    xmax: float
    ymax: float

## edgeとnodeを管理する変数

In [4]:
nodes = {}
edges = {}

## 街路ネットワークの作成

In [5]:
#boundary = c2g.get_boundaries("Sapporo, JP")
boundary = BBox(140.98, 42.78, 141.51, 43.18)

In [6]:
road = c2g.load_overture_data(
    #area=boundary.geometry.iloc[0],
    area=boundary,
    types=["segment", "connector"],
    save_to_file=False
)
road_segment = road['segment'].to_crs(epsg=6691)
road_connector = road['connector'].to_crs(epsg=6691)

In [7]:
segments = c2g.process_overture_segments(
    road_segment,
    connectors_gdf=road_connector,
    get_barriers=True,
    threshold=1.0
)

In [8]:
street_nodes, street_edges = c2g.segments_to_graph(segments)
edges[("intersection", "walks_to", "intersection")] = street_edges
nodes['intersection'] = street_nodes

Removed 27 invalid geometries


## 図書館ノードの作成

## overpass APIを使ってデータを収集する方法

In [9]:
library_tag = {
    "amenity": ["library"]
}

libraries_gdf = ox.features_from_bbox(
    #boundary.total_bounds, # sapporo
    boundary,
    tags=library_tag
)

libraries_gdf.geometry = libraries_gdf.geometry.representative_point()
libraries_gdf = libraries_gdf.to_crs(epsg=6691)

KeyboardInterrupt: 

## layercakeによってデータを得る方法

In [10]:
conn = duckdb.connect()
conn.execute("INSTALL spatial; LOAD spatial;")

layercake_url = "https://data.openstreetmap.us/layercake/pois.parquet"

query = f"""
SELECT
  type,
  id,
  name,
  operator,
  wikidata,
  geometry
FROM '{layercake_url}'
WHERE bbox.xmax >= {boundary.xmin}
  AND bbox.xmin <= {boundary.xmax}
  AND bbox.ymax >= {boundary.ymin}
  AND bbox.ymin <= {boundary.ymax}
  AND ST_Intersects(geometry, ST_MakeEnvelope({boundary.xmin}, {boundary.ymin}, {boundary.xmax}, {boundary.ymax}))
  AND "amenity" = 'library';
"""
libraries_df = conn.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
libraries_df['geometry'] = libraries_df['geometry'].apply(lambda x: wkb.loads(bytes(x)))
libraries_gdf = gpd.GeoDataFrame(libraries_df, geometry='geometry', crs="EPSG:4326")
libraries_gdf.geometry = libraries_gdf.geometry.representative_point()
libraries_gdf = libraries_gdf.to_crs(epsg=6691)

In [14]:
nodes['libraries'] = libraries_gdf

In [21]:
# nearest_points_from_libraries = libraries_gdf.sjoin_nearest(
#     processed_roads,
#     how="left",
#     distance_col="access_dist",
# )
print(road["segment"].crs)
print(segments.crs)
print(segments.geometry.geom_type.value_counts())
print("rows:", len(segments))
print("empty:", segments.geometry.is_empty.sum())
print("invalid:", (~segments.geometry.is_valid).sum())
print("zero length:", (segments.geometry.length == 0).sum())

EPSG:4326
EPSG:6691
LineString    246576
Name: count, dtype: int64
rows: 246576
empty: 0
invalid: 27
zero length: 27


## 図書館と街路とを接続するedgeの作成

In [15]:
_, access_edge = c2g.bridge_nodes(
    nodes,
    proximity_method="knn",
    source_node_types=['libraries'],
    target_node_types=['intersection'],
    k=1,
    distance_metric="euclidean",
)
edges.update(access_edge)

In [31]:
from pyproj import Transformer
import shapely

center = Point([ 141.3399803, 43.0300989 ])
transformer = Transformer.from_crs(4326, 6691, always_xy=True)
center_metered = shapely.transform(center, transformer.transform, interleaved=False)
shapely.to_geojson(center)

'{"type":"Point","coordinates":[141.3399803,43.0300989]}'

In [ ]:
isochrone_gdf = c2g.create_isochrone(
    nodes=nodes,
    edges=edges,
    center_point=center_metered,
    threshold=30 * 60
)

In [33]:
import folium

m = folium.Map()
isochrone_layer = folium.GeoJson(
    isochrone_gdf.to_crs(epsg=4326),
    style_function=lambda _: {
        "color": "#2563eb",
        "weight": 2,
        "fillOpacity": 0.25,
    },
).add_to(m)
library_layer = folium.GeoJson(
    center,
).add_to(m)

m.fit_bounds(isochrone_layer.get_bounds())
m

## GTFSデータの追加
- [ ] GTFSを読み込む
- [ ] node, edgeに追加する

In [62]:
joutetsu_gtfs = c2g.load_gtfs('./joutetsu.zip')

In [63]:
jt_nodes, jt_edges = c2g.travel_summary_graph(
    joutetsu_gtfs,
)
jt_nodes_meter = jt_nodes.to_crs(epsg=6691)
jt_edges_meter = jt_edges.to_crs(epsg=6691)

In [64]:
nodes["jt_stop"] = jt_nodes_meter
edges[("jt_stop", "jotetsu", "jt_stop")] = jt_edges_meter

In [65]:
_, access_edge = c2g.bridge_nodes(
    nodes,
    proximity_method="knn",
    source_node_types=['jt_stop'],
    target_node_types=['intersection'],
    k=1,
    distance_metric="euclidean",
)
edges.update(access_edge)

In [40]:
import folium

m = folium.Map()
jt_node_layer = folium.GeoJson(
    jt_nodes,
).add_to(m)
jt_edge_layer = folium.GeoJson(
    jt_edges,
).add_to(m)

m.fit_bounds(isochrone_layer.get_bounds())
m

## 札幌市交通局

In [41]:
st_gtfs = c2g.load_gtfs('./st.zip')

In [48]:
st_nodes, st_edges = c2g.travel_summary_graph(
    st_gtfs,
)
st_nodes_meter = st_nodes.to_crs(epsg=6691)
st_edges_meter = st_edges.to_crs(epsg=6691)

In [44]:
import folium

m = folium.Map()
st_node_layer = folium.GeoJson(
    st_nodes,
).add_to(m)
st_edge_layer = folium.GeoJson(
    st_edges,
).add_to(m)

m.fit_bounds(isochrone_layer.get_bounds())
m

In [50]:
nodes["st_station"] = st_nodes_meter
edges[("st_station", "subway", "st_station")] = st_edges_meter

In [58]:
_, access_edge = c2g.bridge_nodes(
    nodes,
    proximity_method="knn",
    source_node_types=['st_station'],
    target_node_types=['intersection'],
    k=1,
    distance_metric="euclidean",
)
edges.update(access_edge)

In [73]:
walk_speed_mps = 4.8 / 3.6  # 徒歩4.8 km/hと仮定

for edge_type, gdf in edges.items():
    if "travel_time_sec" not in gdf.columns:
        gdf = gdf.copy()
        gdf["travel_time_sec"] = gdf.geometry.length / walk_speed_mps
        edges[edge_type] = gdf

In [75]:
transit_isochrone_gdf = c2g.create_isochrone(
    nodes=nodes,
    edges=edges,
    center_point=center_metered,
    threshold=30 * 60,
    edge_attr="travel_time_sec"
)

In [76]:
import folium

m = folium.Map()
transit_isochrone_layer = folium.GeoJson(
    transit_isochrone_gdf.to_crs(epsg=4326)
).add_to(m)
library_layer = folium.GeoJson(
    center,
).add_to(m)
#st_layer = folium.GeoJson(
#    edges[('st_station', 'subway', 'st_station')].to_crs(epsg=4326),
#).add_to(m)

m.fit_bounds(transit_isochrone_layer.get_bounds())
m

## GBFSデータの追加
- [ ] GBFSを読み込む
- [ ] node, edgeに追加する